In [1]:
import urllib.request, zipfile, hashlib
# Download
url = "https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip"
urllib.request.urlretrieve(url, "data.zip")
print("Download complete.")
# SHA256
with open("data.zip", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"SHA256: {sha256}")

if sha256 == "9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67":
    print("sha256 matched")
else:
    raise Exception("sha256 not matched")
# Unzip
with zipfile.ZipFile("data.zip", "r") as z:
    z.extractall(".")
print("Unzipped.")

Download complete.
SHA256: 9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67
sha256 matched
Unzipped.


In [1]:
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [2]:
from torchvision import transforms

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [3]:
from torchvision import datasets
evaluation_dataset = datasets.ImageFolder("data/test",transform=evaluation_transform)
train_dataset = datasets.ImageFolder("data/train",transform=evaluation_transform)


In [4]:
from torch.utils.data import DataLoader

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [5]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [6]:
import torch
import torch.optim as optim

device = torch.device("cpu")

model = HairModel()
model.to(device)

HairModel(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

### Question 1 — loss and output

### Answer

nn.BCEWithLogitsLoss()

### Question 2 — parameter count

### Answer

In [8]:
sum(p.numel() for p in model.parameters())

20073473

In [9]:
import json

history = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": [],
}

In [24]:
def train_and_evaluate(model, optimizer, train_loader, evaluation_loader, criterion, num_epochs, device,file_path=None):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * inputs.size(0)
            correct += ((torch.sigmoid(outputs) >= 0.5) == labels).sum().item()
            total += labels.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in evaluation_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                labels = labels.float().unsqueeze(1)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * inputs.size(0)
                val_correct += ((torch.sigmoid(outputs) >= 0.5) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        if file_path:
            history["train_loss"].append(train_loss)
            history["train_accuracy"].append(train_acc)
            history["evaluation_loss"].append(val_loss)
            history["evaluation_accuracy"].append(val_acc)

            with open(file_path, "w") as f:
                json.dump(history, f, indent=4)

        # if val_acc > best_val_accuracy:
        #   best_val_accuracy = val_acc
        #   checkpoint_path = f'mobilenet_v4_{epoch+1:02d}_{val_acc:.3f}.pth'
        #   torch.save(model.state_dict(), checkpoint_path)
        #   print(f'Checkpoint saved: {checkpoint_path}')

### Train for 10 epochs with the baseline transform. For every epoch, evaluate the held-out split with model.eval() and torch.no_grad(). Store these four series in a JSON file named history_baseline.json:

In [ ]:
train_and_evaluate(model, optimizer, train_loader, evaluation_loader, criterion, 10, device,"history_baseline.json")

Epoch 1/10
  Train Loss: 0.6277, Train Acc: 0.6637
  Val Loss: 0.5987, Val Acc: 0.6716
Epoch 2/10
  Train Loss: 0.5531, Train Acc: 0.7163
  Val Loss: 0.5837, Val Acc: 0.6517
Epoch 3/10
  Train Loss: 0.5055, Train Acc: 0.7425
  Val Loss: 0.5704, Val Acc: 0.6766
Epoch 4/10
  Train Loss: 0.4366, Train Acc: 0.7975
  Val Loss: 0.6199, Val Acc: 0.6915
Epoch 5/10
  Train Loss: 0.4383, Train Acc: 0.7963
  Val Loss: 0.5933, Val Acc: 0.7015
Epoch 6/10
  Train Loss: 0.3134, Train Acc: 0.8700
  Val Loss: 0.5962, Val Acc: 0.6816
Epoch 7/10
  Train Loss: 0.2777, Train Acc: 0.8888
  Val Loss: 0.7215, Val Acc: 0.6567
Epoch 8/10
  Train Loss: 0.2505, Train Acc: 0.8962
  Val Loss: 0.6481, Val Acc: 0.7313
Epoch 9/10
  Train Loss: 0.1527, Train Acc: 0.9487
  Val Loss: 0.8784, Val Acc: 0.6716
Epoch 10/10
  Train Loss: 0.2116, Train Acc: 0.9113
  Val Loss: 0.6135, Val Acc: 0.7413


### Question 3 — baseline accuracy

In [16]:
import json
import numpy as np

with open("history_baseline.json") as f:
    history = json.load(f)

median = np.median(history["train_accuracy"])
print(round(float(median), 2))

0.83


### Question 4 — baseline loss

In [20]:
import json
import numpy as np

with open("history_baseline.json") as f:
    history = json.load(f)

std = np.std(history["train_loss"],ddof=0)
print(round(float(std), 3))

0.15


In [21]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
train_dataset_augmented = datasets.ImageFolder("data/train",train_transform_augmented)

In [25]:
train_augmented_loader= DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

In [26]:
train_and_evaluate(model, optimizer, train_augmented_loader, evaluation_loader, criterion, 10, device,"history_augmented.json")

Epoch 1/10
  Train Loss: 0.6794, Train Acc: 0.6012
  Val Loss: 0.6195, Val Acc: 0.6418
Epoch 2/10
  Train Loss: 0.6279, Train Acc: 0.6425
  Val Loss: 0.6599, Val Acc: 0.6368
Epoch 3/10
  Train Loss: 0.6039, Train Acc: 0.6587
  Val Loss: 0.5981, Val Acc: 0.6567
Epoch 4/10
  Train Loss: 0.5911, Train Acc: 0.6675
  Val Loss: 0.6062, Val Acc: 0.6617
Epoch 5/10
  Train Loss: 0.5988, Train Acc: 0.6450
  Val Loss: 0.5848, Val Acc: 0.6617
Epoch 6/10
  Train Loss: 0.5947, Train Acc: 0.6600
  Val Loss: 0.5804, Val Acc: 0.6667
Epoch 7/10
  Train Loss: 0.5731, Train Acc: 0.6775
  Val Loss: 0.5886, Val Acc: 0.6567
Epoch 8/10
  Train Loss: 0.5770, Train Acc: 0.6637
  Val Loss: 0.5771, Val Acc: 0.6617
Epoch 9/10
  Train Loss: 0.5680, Train Acc: 0.6763
  Val Loss: 0.5834, Val Acc: 0.6866
Epoch 10/10
  Train Loss: 0.5615, Train Acc: 0.6913
  Val Loss: 0.5741, Val Acc: 0.6667


### Question 5 — augmented evaluation loss

In [27]:
import json
import numpy as np

with open("history_augmented.json") as f:
    history = json.load(f)

mean = np.mean(history["evaluation_loss"])
print(round(float(mean), 3))

0.62


### Question 6 — augmented evaluation accuracy

In [29]:
import json
import numpy as np

with open("history_augmented.json") as f:
    history = json.load(f)

mean = np.mean(history["evaluation_accuracy"][5:10])
print(round(float(mean), 2))

0.7
